In [ ]:
# GLOBAL #

import random
import pandas as pd
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from typing import Literal
import os
import string

random.seed(123)
np.random.seed(123)
random_state = 123

In [ ]:
# CREATE CORPORA #

# function to create dom/sub/full (combined) corpora by species
def create_corpus_by_species(df, name: str, by_length=True):

    #for each species:
    for species, species_df in df.groupby('species'):
        dom_filename  = f"corpus_{species}_{name}_dom.txt" #dom file
        sub_filename  = f"corpus_{species}_{name}_sub.txt" #sub file
        full_filename = f"corpus_{species}_{name}_full.txt" #full file

        #group df by trial and sequence
        grouped = species_df.groupby(['trial_id', 'sequence_id'])

        with open(dom_filename, 'w') as f, open(sub_filename, 'w') as g, open(full_filename, 'w') as h:
            for (trial_id, seq_id), group in grouped:
                #dom, then sub
                for idx, fish_role in enumerate(['dom', 'sub']):
                    role_group = group[group['fish_id'] == fish_role].sort_values('order_id')
                    sequence_parts = []
                    for _, row in role_group.iterrows():

                        #by_length repetition
                        if by_length:
                            chars = [str(row['state'])] * int(row['length'])
                        else:
                            chars = [str(row['state'])]

                        sequence_parts.append(''.join(chars))

                    #add bos and eos tokens
                    sequence_line = "<s>" + ''.join(sequence_parts) + '</s>\n'

                    if idx == 0: #if dom
                        f.write(sequence_line)
                    else: #else sub
                        g.write(sequence_line)

                    h.write(sequence_line)


#start with full dataset (all species)
dat = pd.read_csv('PrettyDat.csv')

# train and test lists
train_list = []
test_list = []

# Group by species and split by trial_id into train and test
for species, group in dat.groupby("species"):
    trial_ids = group["trial_id"].unique()
    train_trials, test_trials = train_test_split(
        trial_ids, test_size = 0.2, random_state=random_state
    )
    
    train_subset = group[group["trial_id"].isin(train_trials)] #because corresponding dom and sub seq will have same trial num, they will automatically be kept together
    test_subset = group[group["trial_id"].isin(test_trials)]
    
    train_list.append(train_subset)
    test_list.append(test_subset)

# Combine all species back
train = pd.concat(train_list).reset_index(drop=True)
test = pd.concat(test_list).reset_index(drop=True)

#check whether sizes match
for species in dat["species"].unique():
    print(species)
    full_temp = dat[dat['species'] == species]
    train_temp = train[train['species'] == species]
    test_temp = test[test['species'] == species]
    print(len(full_temp['trial_id'].unique()))
    print(len(train_temp['trial_id'].unique()))
    print(len(test_temp['trial_id'].unique()))
    assert len(train_temp['trial_id'].unique()) + len(test_temp['trial_id'].unique()) == len(full_temp['trial_id'].unique())

#create each corpus after splitting full into train and test
create_corpus_by_species(train, name = "train", by_length=True)
create_corpus_by_species(test, name = "test", by_length=True)
